# HELD: BCC force constants through 5NN, saved by MD step

This notebook scans the complete non-magnetic iron dataset under `IronCoreMD/dataset/bcc/non-mag`, fits every compatible Fe BCC trajectory with the repository's HELD implementation using five BCC neighbor shells, and saves one compressed NPZ per trajectory plus a master index.

For every MD step, the output stores the 14 traditional monoatomic-BCC Born–von Kármán elements through 5NN: `alpha_0`, `alpha_1`, `beta_1`, `alpha_2`, `beta_2`, `alpha_3`, `beta_3`, `gamma_3`, `alpha_4`, `beta_4`, `gamma_4`, `delta_4`, `alpha_5`, and `beta_5`.

In [8]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd().resolve()
if not (NOTEBOOK_DIR / 'held_bcc_5nn.py').is_file():
    raise FileNotFoundError('Run this notebook from IronCoreMD/ForceConstants')
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from held_bcc_5nn import (
    BVK_LABELS, DATASET_ROOT, RESULTS_ROOT, FitConfig,
    discover_bcc_npz, fit_trajectory, run_dataset, summarize_index,
)

print('Dataset:', DATASET_ROOT)
print('Results:', RESULTS_ROOT)
print('14 BvK labels:', BVK_LABELS.tolist())

Dataset: /Users/dajuarez4/Documents/Fe/IronCoreMD/dataset/bcc/non-mag
Results: /Users/dajuarez4/Documents/Fe/IronCoreMD/ForceConstants/results
14 BvK labels: ['alpha_0', 'alpha_1', 'beta_1', 'alpha_2', 'beta_2', 'alpha_3', 'beta_3', 'gamma_3', 'alpha_4', 'beta_4', 'gamma_4', 'delta_4', 'alpha_5', 'beta_5']


## 1. Discover and audit non-magnetic Fe BCC NPZ files

Only NPZ files inside `dataset/bcc/non-mag` with the complete HELD trajectory schema and elemental Fe labels are selected.

In [9]:
trajectories, skipped = discover_bcc_npz()
inventory = pd.DataFrame({
    'relative_path': [str(path.relative_to(DATASET_ROOT)) for path in trajectories],
    'size_MB': [path.stat().st_size / 1024**2 for path in trajectories],
})
print(f'Compatible trajectories: {len(trajectories)}')
print(f'Skipped NPZ files: {len(skipped)}')
display(inventory)
display(pd.DataFrame(skipped))

Compatible trajectories: 138
Skipped NPZ files: 0


,relative_path,size_MB
0,2.29_4000K.npz,1.086247
1,2.29_4500K.npz,1.086048
2,2.29_5000K.npz,1.085899
3,2.29_5500K.npz,1.086541
4,2.29_6000K.npz,1.086326
...,...,...
133,2.55_4000K.npz,1.087907
134,2.55_4500K.npz,1.088533
135,2.55_5000K.npz,1.088635
136,2.55_5500K.npz,0.445948


""


## 2. Smoke test on one real trajectory

This fits two finite frames and validates the result schema before starting the full dataset. It writes into `results/smoke/` and does not overwrite production outputs.

In [10]:
smoke_source = next(
    (path for path in trajectories if 'non-mag' in path.parts),
    trajectories[0],
)
smoke = fit_trajectory(
    smoke_source,
    config=FitConfig(skip=0, every=1, max_frames=2, overwrite=True),
    results_root=RESULTS_ROOT / 'smoke',
    verbose=True,
)
smoke

[HELD] solved frame 1/2 (step=1)
[HELD] solved frame 2/2 (step=2)


{'case_id': '2_29_4000K__43ee4c4f',
 'source_path': '/Users/dajuarez4/Documents/Fe/IronCoreMD/dataset/bcc/non-mag/2.29_4000K.npz',
 'output_path': '/Users/dajuarez4/Documents/Fe/IronCoreMD/ForceConstants/results/smoke/2_29_4000K__43ee4c4f_held_5nn.npz',
 'status': 'completed',
 'n_frames': 2,
 'natoms': 128,
 'fc_mean': array([ 54.63229276,  -4.3631839 , -10.77095064,  -1.56181914,
          0.33679525,  -1.02245807,  -0.17079581,  -0.77729683,
         -0.79123612,  -0.34780322,   0.48002914,  -0.06059724,
         -0.07492812,  -0.48054374]),
 'shell_distances_ang': array([1.98319814, 2.28999996, 3.238549  , 3.79753532, 3.96639628]),
 'elapsed_s': 0.3535703329980606}

In [11]:
with np.load(smoke['output_path'], allow_pickle=False) as result:
    expected = ['alpha_0', 'alpha_1', 'beta_1', 'alpha_2', 'beta_2', 'alpha_3', 'beta_3', 'gamma_3', 'alpha_4', 'beta_4', 'gamma_4', 'delta_4', 'alpha_5', 'beta_5']
    assert result['fc_labels'].tolist() == expected
    assert result['fc_per_md_step'].shape == (2, 14)
    assert result['step_ids'].shape == (2,)
    assert result['held_coefficients_per_frame'].shape[0] == 2
    assert result['offsite_fc_per_frame'].shape[0] == 2
    assert result['onsite_fc_per_frame'].shape[0] == 2
    display(pd.DataFrame(
        result['fc_per_md_step'],
        index=result['step_ids'],
        columns=result['fc_labels'],
    ).rename_axis('md_step'))
print('PASS: per-MD-step arrays and alpha labels validated')

,alpha_0,alpha_1,beta_1,alpha_2,beta_2,alpha_3,beta_3,gamma_3,alpha_4,beta_4,gamma_4,delta_4,alpha_5,beta_5
md_step,,,,,,,,,,,,,,
1,54.867644,-4.298054,-10.822664,-1.482917,0.352451,-1.006315,-0.179177,-0.809109,-0.888156,-0.355988,0.467695,-0.071602,-0.083432,-0.505828
2,54.396942,-4.428314,-10.719238,-1.640721,0.321139,-1.038601,-0.162415,-0.745485,-0.694316,-0.339619,0.492363,-0.049593,-0.066424,-0.455260


PASS: per-MD-step arrays and alpha labels validated


## 3. Full non-magnetic Fe BCC dataset fit

Set `RUN_FULL_DATASET = True` when ready. The default configuration uses every finite MD frame and all five shells. This is computationally expensive because the repository contains many 128-atom, 400-frame trajectories. Completed case files are reused on restart.

To discard equilibration frames, change `skip`; to thin correlated frames, change `every`. Those selections are stored in every result file.

In [12]:
RUN_FULL_DATASET = True  # Change to True to launch all compatible BCC trajectories.
CONFIG = FitConfig(
    skip=0,
    every=1,
    max_frames=0,  # 0 means all selected frames
    aggregate='mean',
    num_shells=5,
    overwrite=False,  # resume safely from existing per-case results
)

if RUN_FULL_DATASET:
    records, skipped, master_index = run_dataset(config=CONFIG, verbose=True)
    print('Master index:', master_index)
    summarize_index(master_index)
else:
    print('Dry run only. Set RUN_FULL_DATASET=True to fit the full BCC dataset.')

[dataset 1/138] 2.29_4000K.npz
[HELD] solved frame 1/399 (step=1)
[HELD] solved frame 25/399 (step=25)
[HELD] solved frame 50/399 (step=50)
[HELD] solved frame 75/399 (step=75)
[HELD] solved frame 100/399 (step=100)
[HELD] solved frame 125/399 (step=125)
[HELD] solved frame 150/399 (step=150)
[HELD] solved frame 175/399 (step=175)
[HELD] solved frame 200/399 (step=200)
[HELD] solved frame 225/399 (step=225)
[HELD] solved frame 250/399 (step=250)
[HELD] solved frame 275/399 (step=275)
[HELD] solved frame 300/399 (step=300)
[HELD] solved frame 325/399 (step=325)
[HELD] solved frame 350/399 (step=350)
[HELD] solved frame 375/399 (step=375)
[HELD] solved frame 399/399 (step=399)
[dataset 2/138] 2.29_4500K.npz
[HELD] solved frame 1/399 (step=1)
[HELD] solved frame 25/399 (step=25)
[HELD] solved frame 50/399 (step=50)
[HELD] solved frame 75/399 (step=75)
[HELD] solved frame 100/399 (step=100)
[HELD] solved frame 125/399 (step=125)
[HELD] solved frame 150/399 (step=150)
[HELD] solved frame 17

## 4. Inspect the master index and per-step results

The master index stores one row per attempted trajectory. Each `output_path` points to a detailed NPZ containing the step-resolved arrays.

In [13]:
master_index = RESULTS_ROOT / 'bcc_held_5nn_index.npz'
if master_index.exists():
    with np.load(master_index, allow_pickle=False) as index:
        index_table = pd.DataFrame({
            'case_id': index['case_id'],
            'status': index['status'],
            'n_frames': index['n_frames'],
            'natoms': index['natoms'],
            **{label: index['fc_mean'][:, i] for i, label in enumerate(index['fc_labels'])},
            'output_path': index['output_path'],
            'error': index['error'],
        })
    display(index_table)
else:
    print('The full-dataset index does not exist yet.')

,case_id,status,n_frames,natoms,alpha_0,alpha_1,beta_1,alpha_2,beta_2,alpha_3,beta_3,gamma_3,alpha_4,beta_4,gamma_4,delta_4,alpha_5,beta_5,output_path,error
0,2_29_4000K__43ee4c4f,completed,399,128,47.234569,-4.108108,-7.593066,-6.410474,0.963980,-0.408366,0.424752,-0.459035,-0.792778,0.135793,0.059252,0.396112,-0.374962,-0.617593,/Users/dajuarez4/Documents/Fe/IronCoreMD/Force...,
1,2_29_4500K__ed36d84b,completed,399,128,46.390022,-3.744209,-7.613722,-7.179695,1.076894,-0.447009,-0.031747,-0.538812,-0.515098,0.111071,0.083949,0.231798,-0.249860,-0.455447,/Users/dajuarez4/Documents/Fe/IronCoreMD/Force...,
2,2_29_5000K__e8b3ab4b,completed,399,128,46.066560,-3.643656,-7.497555,-6.543291,0.508757,-0.447800,0.217477,-0.553989,-0.855490,0.047517,0.082515,0.177492,0.033659,-0.526690,/Users/dajuarez4/Documents/Fe/IronCoreMD/Force...,
3,2_29_5500K__433f7228,completed,399,128,45.988021,-3.727766,-7.368307,-6.764833,0.870699,-0.385962,-0.043647,-0.574281,-0.520961,0.103686,0.062557,0.253007,-0.214660,-0.492502,/Users/dajuarez4/Documents/Fe/IronCoreMD/Force...,
4,2_29_6000K__56191404,completed,399,128,46.394093,-3.835153,-7.592664,-7.243321,1.084347,-0.231174,0.132619,-0.557044,-0.686045,-0.005594,0.032933,0.123972,-0.015252,-0.433490,/Users/dajuarez4/Documents/Fe/IronCoreMD/Force...,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133,2_55_4000K__3b01b60f,completed,399,128,17.417398,-1.315502,-2.769640,-2.613679,0.493079,-0.103547,-0.121677,-0.179851,-0.183074,0.009734,-0.017037,-0.029915,-0.117735,-0.176789,/Users/dajuarez4/Documents/Fe/IronCoreMD/Force...,
134,2_55_4500K__f2bb4b5e,completed,399,128,12.896140,-0.912486,-2.153356,-2.181492,0.796572,-0.140996,-0.259890,-0.262340,-0.180281,-0.025808,-0.022634,-0.058323,0.009842,-0.161101,/Users/dajuarez4/Documents/Fe/IronCoreMD/Force...,
135,2_55_5000K__49ac84d1,completed,399,128,9.171561,-0.706299,-1.491510,-1.372231,0.645213,0.046483,-0.111365,-0.240374,-0.237977,-0.026934,-0.045668,-0.034241,-0.039726,-0.038941,/Users/dajuarez4/Documents/Fe/IronCoreMD/Force...,
136,2_55_5500K__4a3c421d,completed,162,128,13.681237,-0.964169,-2.284323,-2.054882,0.521578,0.084360,-0.297724,-0.302000,-0.133028,0.019766,-0.032280,0.010798,-0.144013,-0.130063,/Users/dajuarez4/Documents/Fe/IronCoreMD/Force...,


In [14]:
# Example: load one completed case and make an MD-step table.
if master_index.exists():
    completed = index_table[index_table.status.isin(['completed', 'cached'])]
    if len(completed):
        selected_output = Path(completed.iloc[0].output_path)
        with np.load(selected_output, allow_pickle=False) as result:
            step_table = pd.DataFrame(
                result['fc_per_md_step'],
                index=result['step_ids'],
                columns=result['fc_labels'],
            ).rename_axis('md_step')
            display(step_table)
            print('Units:', result['fc_units'].item())
            print('Shell distances (angstrom):', result['shell_distances_ang'])

,alpha_0,alpha_1,beta_1,alpha_2,beta_2,alpha_3,beta_3,gamma_3,alpha_4,beta_4,gamma_4,delta_4,alpha_5,beta_5
md_step,,,,,,,,,,,,,,
1,54.867644,-4.298054,-10.822664,-1.482917,0.352451,-1.006315,-0.179177,-0.809109,-0.888156,-0.355988,0.467695,-0.071602,-0.083432,-0.505828
2,54.396942,-4.428314,-10.719238,-1.640721,0.321139,-1.038601,-0.162415,-0.745485,-0.694316,-0.339619,0.492363,-0.049593,-0.066424,-0.455260
3,46.607151,-4.351740,-10.645867,-1.418268,0.531397,-0.918803,-0.035723,-0.720246,-0.471282,-0.215306,0.485627,-0.039111,0.011734,-0.444446
4,46.761258,-4.493421,-10.675140,-1.643079,0.608384,-0.915598,-0.047740,-0.687962,-0.308403,-0.205457,0.478413,-0.020540,-0.020302,-0.412657
5,46.661588,-4.586441,-10.623059,-1.843943,0.699014,-0.872066,-0.057791,-0.657701,-0.190054,-0.185630,0.461331,-0.000913,-0.079640,-0.399224
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,51.770588,-4.815662,-7.776462,-8.591167,0.778761,1.096913,0.564629,-0.275132,-0.210603,0.091866,0.030153,1.080153,-0.983464,-0.907273
396,51.475718,-4.784090,-7.722408,-8.597828,0.932906,1.105603,0.512118,-0.361389,-0.211989,0.065727,0.039219,1.051963,-0.951754,-0.913123
397,51.057999,-4.752442,-7.661351,-8.574564,1.001098,1.100090,0.478438,-0.393856,-0.195142,0.043656,0.051292,1.024189,-0.907370,-0.906723


Units: eV/angstrom^2
Shell distances (angstrom): [1.98319814 2.28999996 3.238549   3.79753532 3.96639628]
